In [6]:
#data structures
import pandas as pd
import numpy as np

#machine learning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler

from sklearn.pipeline import Pipeline

#metrics (performace and machine learning scores)
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import time #used for seeing how long it takes to run programs

np.random.seed(42)

In [7]:
import os
print(os.getcwd())

D:\Code\Earthquake


In [8]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

In [30]:
#Sampling the data so test runs don't take so long
dataSample = df.sample(frac=1)
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')

#Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade


features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
stringTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']

one_hot_tables = pd.get_dummies(features[stringTables], prefix='education', dtype=int)


# scaler = MinMaxScaler()
# X = scaler.fit_transform(features)

#Now we replace our string tables with the encoded tables
X = features.drop(stringTables, axis = 1)
X = X.join(one_hot_tables)


y = dataSample['damage_grade']

In [40]:
dataSample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 260601 entries, 128570 to 122257
Data columns (total 40 columns):
 #   Column                                  Non-Null Count   Dtype 
---  ------                                  --------------   ----- 
 0   building_id                             260601 non-null  int64 
 1   geo_level_1_id                          260601 non-null  int64 
 2   geo_level_2_id                          260601 non-null  int64 
 3   geo_level_3_id                          260601 non-null  int64 
 4   count_floors_pre_eq                     260601 non-null  int64 
 5   age                                     260601 non-null  int64 
 6   area_percentage                         260601 non-null  int64 
 7   height_percentage                       260601 non-null  int64 
 8   land_surface_condition                  260601 non-null  object
 9   foundation_type                         260601 non-null  object
 10  roof_type                               260601 non-null 

In [31]:
#grid search pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': range(1, 11), # Test k from 1 to 20
    'knn__weights': ['distance'], # Test different weight functions
    'knn__metric': ['manhattan'] # Test different distance metrics
}

#This code was taken from the documentation
gridSearch = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1 # Use all available processors
)

In [32]:
gridSearch.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'knn__metric': ['manhattan'], 'knn__n_neighbors': range(1, 11), 'knn__weights': ['distance']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,feature_range,"(0, ...)"


In [33]:
print(f"Best parameters: {gridSearch.best_params_}")
print(f"Best cross-validation score: {gridSearch.best_score_:.4f}")

# Get the best model
best_knn_model = gridSearch.best_estimator_

Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 10, 'knn__weights': 'distance'}
Best cross-validation score: 0.6749


In [34]:
test_accuracy = best_knn_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")

Test set accuracy with best model: 0.6811


In [35]:
param_grid = {
    'knn__n_neighbors': range(11, 21), # Test k from 1 to 20
    'knn__weights': ['distance'], # Test different weight functions
    'knn__metric': ['manhattan'] # Test different distance metrics
}

#This code was taken from the documentation
gridSearch2 = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1, # Use all available processors
)

In [36]:
gridSearch2.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'knn__metric': ['manhattan'], 'knn__n_neighbors': range(11, 21), 'knn__weights': ['distance']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,feature_range,"(0, ...)"


In [37]:
print(f"Best parameters: {gridSearch2.best_params_}")
print(f"Best cross-validation score: {gridSearch2.best_score_:.4f}")

# Get the best model
best_knn_model = gridSearch2.best_estimator_

Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 20, 'knn__weights': 'distance'}
Best cross-validation score: 0.6785


In [38]:
print(f"Best parameters: {gridSearch2.best_params_}")
print(f"Best cross-validation score: {gridSearch2.best_score_:.4f}")

# Get the best model
best_knn_model = gridSearch2.best_estimator_

Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 20, 'knn__weights': 'distance'}
Best cross-validation score: 0.6785


In [39]:
test_accuracy = best_knn_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")

Test set accuracy with best model: 0.6852


In [ ]:
gscv = GridSearchCV(estimator, param_grid, cv=3, verbose=2)